<a href="https://colab.research.google.com/github/nehasingh913833-gif/basic-ML/blob/main/Basic_ml_opertations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

using german credit dataset
1. inverstigate whether attributes such asa foreign worker and personal status introcduce bias into the prediction model
2. perform preprocessing  and feature selection
3. train decision tree and random forest models using cross validation
4. compare model performances before and after removing the sensitive attributes
5. analyse the impact on fairness and predivtive performance


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))



Saving German_Credit_data.csv to German_Credit_data.csv
User uploaded file "German_Credit_data.csv" with length 48050 bytes


In [ ]:
df = pd.read_csv('German_Credit_data.csv')
print(df.info())
print(df.head())
print("Columns:", df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   status                   1000 non-null   int64
 1   duration                 1000 non-null   int64
 2   credit_history           1000 non-null   int64
 3   purpose                  1000 non-null   int64
 4   amount                   1000 non-null   int64
 5   savings                  1000 non-null   int64
 6   employment_duration      1000 non-null   int64
 7   installment_rate         1000 non-null   int64
 8   personal_status_sex      1000 non-null   int64
 9   other_debtors            1000 non-null   int64
 10  present_residence        1000 non-null   int64
 11  property                 1000 non-null   int64
 12  age                      1000 non-null   int64
 13  other_installment_plans  1000 non-null   int64
 14  housing                  1000 non-null   int64
 15  numbe

In [ ]:
print("Credit Risk values:", df['credit_risk'].unique())
print("Foreign Worker values:", df['foreign_worker'].unique())
print("Personal Status Sex values:", df['personal_status_sex'].unique())

Credit Risk values: [1 0]
Foreign Worker values: [2 1]
Personal Status Sex values: [2 3 4 1]


In [ ]:

# Data
X = df.drop(columns=['credit_risk'])
y = df['credit_risk']

# Sensitive attributes
sensitive_attrs = ['foreign_worker', 'personal_status_sex']
X_unbiased = X.drop(columns=sensitive_attrs)

# Models
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model_and_fairness(model, X_data, df_original):
    # Predict using CV
    preds = cross_val_predict(model, X_data, y, cv=cv)

    # Predictive metrics
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)

    # Fairness metrics
    # We use df_original to keep track of sensitive groups regardless of whether they were in X_data
    df_eval = df_original.copy()
    df_eval['pred'] = preds

    # Disparate Impact / Demographic Parity for Foreign Worker
    rates_fw = df_eval.groupby('foreign_worker')['pred'].mean()
    dp_fw = rates_fw.max() - rates_fw.min()

    # Disparate Impact / Demographic Parity for Personal Status Sex
    rates_ps = df_eval.groupby('personal_status_sex')['pred'].mean()
    dp_ps = rates_ps.max() - rates_ps.min()

    return {
        'Accuracy': acc,
        'F1 Score': f1,
        'DP Diff (Foreign Worker)': dp_fw,
        'DP Diff (Personal Status)': dp_ps,
        'Positive Rate FW': rates_fw.to_dict(),
        'Positive Rate PS': rates_ps.to_dict()
    }

print("--- With Sensitive Attributes ---")
res_dt_biased = evaluate_model_and_fairness(dt, X, df)
res_rf_biased = evaluate_model_and_fairness(rf, X, df)
print("Decision Tree:", res_dt_biased)
print("Random Forest:", res_rf_biased)

print("\n--- Without Sensitive Attributes ---")
res_dt_unbiased = evaluate_model_and_fairness(dt, X_unbiased, df)
res_rf_unbiased = evaluate_model_and_fairness(rf, X_unbiased, df)
print("Decision Tree:", res_dt_unbiased)
print("Random Forest:", res_rf_unbiased)

--- With Sensitive Attributes ---
Decision Tree: {'Accuracy': 0.704, 'F1 Score': 0.7842565597667639, 'DP Diff (Foreign Worker)': 0.0599477982655553, 'DP Diff (Personal Status)': 0.10652173913043483, 'Positive Rate FW': {1: 0.7297297297297297, 2: 0.6697819314641744}, 'Positive Rate PS': {1: 0.6, 2: 0.632258064516129, 3: 0.6952554744525548, 4: 0.7065217391304348}}
Random Forest: {'Accuracy': 0.75, 'F1 Score': 0.8339973439575034, 'DP Diff (Foreign Worker)': 0.08919199573405179, 'DP Diff (Personal Status)': 0.09298737727910233, 'Positive Rate FW': {1: 0.8918918918918919, 2: 0.8026998961578401}, 'Positive Rate PS': {1: 0.76, 2: 0.7548387096774194, 3: 0.8321167883211679, 4: 0.8478260869565217}}

--- Without Sensitive Attributes ---
Decision Tree: {'Accuracy': 0.679, 'F1 Score': 0.764490095377843, 'DP Diff (Foreign Worker)': 0.06929359265807866, 'DP Diff (Personal Status)': 0.1591304347826087, 'Positive Rate FW': {1: 0.7297297297297297, 2: 0.660436137071651}, 'Positive Rate PS': {1: 0.58, 2: 